In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy 
from pfapack import pfaffian as pf


We assume the form 

$$X = e^{\frac{1}{4}\gamma^T A \gamma}$$

which is characterized by 

$$\eta, B$$

In [ ]:
def get_eta_B(A):

    B = scipy.linalg.expm(-A)

    sinhA = scipy.linalg.sinhm(A/4)

    bdim = B.shape[0]

    M = np.zeros((2*bdim, 2*bdim), dtype = np.complex128)

    M[0:bdim, 0:bdim] = np.sqrt(2) * sinhA
    M[bdim:,0:bdim]   = np.identity(bdim)
    M[0:bdim,bdim:]   = -np.identity(bdim)
    M[bdim:,bdim:]    = np.sqrt(2) * sinhA

    eta = np.power(-1,bdim//2) * pf.pfaffian(M)

    return eta, B

def multiplication(eta1, B1, eta2, B2):

    bdim = B1.shape[0]

    M = np.zeros((2*bdim, 2*bdim), dtype = np.complex128)

    G1 = (np.identity(bdim) - B1) @ scipy.linalg.inv(np.identity(bdim) + B1)
    G2 = (np.identity(bdim) - B2) @ scipy.linalg.inv(np.identity(bdim) + B2)

    M[0:bdim, 0:bdim] = G1
    M[bdim:,0:bdim]   = np.identity(bdim)
    M[0:bdim,bdim:]   = -np.identity(bdim)
    M[bdim:,bdim:]    = G2

    eta12 = np.power(-1,bdim//2) * eta1 * eta2 * pf.pfaffian(M)

    B12   = B1@B2

    return eta12, B12

def recursive_eta(Alist):

    Ntau = len(Alist)
    N    = Alist[0].shape[0]//2

    eta, B = get_eta_B(Alist[0])

    eta_prod = eta
    B_prod = B

    for i in range(Ntau-1):

        next_eta, next_B = get_eta_B(Alist[i+1])

        eta_prod, B_prod = multiplication(eta_prod, B_prod, next_eta, next_B)

    return eta_prod*np.power(2, N)

def brute_force_trace(Alist):

    Ntau = len(Alist)
    N    = Alist[0].shape[0] // 2


    eta_prod = 1
    etas = np.zeros(Ntau, dtype=np.complex128)
    Bs   = np.zeros((2*N, 2*N, Ntau), dtype=np.complex128)
    Gs   = np.zeros((2*N, 2*N, Ntau), dtype=np.complex128)

    M    = np.zeros((2*N*Ntau, 2*N*Ntau), dtype=np.complex128)

    I = np.eye(2*N, dtype=np.complex128)

    # Fill M in 2N x 2N blocks
    for i in range(Ntau):
        for j in range(i + 1, Ntau):
            # checkerboard on the upper-right triangle, starting with "-"
            sgn = -1 if ((i + j) % 2 == 1) else 1

            r_i = slice(2*N*i, 2*N*(i + 1))
            r_j = slice(2*N*j, 2*N*(j + 1))

            M[r_i, r_j] = sgn * I          # upper-right block
            M[r_j, r_i] = -sgn * I         # lower-left block (antisymmetric)


    for i in range(Ntau):
        etas[i], Bs[:, :, i] = get_eta_B(Alist[i])
        Gs[:, :, i]          = scipy.linalg.tanhm(Alist[i] / 2)
        eta_prod *= etas[i]
        M[i*2*N:(i+1)*2*N,i*2*N:(i+1)*2*N] = Gs[:,:,i]

    W = np.power(-1, N*Ntau//2) * np.power(2, N) * eta_prod * pf.pfaffian(M)

    return W

def catterpillar(Alist):

    Ntau = len(Alist)
    N    = Alist[0].shape[0] // 2

    temp = np.zeros((4*N, 4*N), dtype = np.complex128)

    Bs = np.zeros((2*N, 2*N, Ntau), dtype = np.complex128)
    etas = np.zeros(Ntau, dtype = np.complex128)
    eta_prod = 1
    pf_prod  = 1

    current_Bstring = np.identity(2*N, dtype=np.complex128)

    for i in range(Ntau):

        etas[i], Bs[:,:,i] = get_eta_B(Alist[i])
        eta_prod *= etas[i]

    for i in range(Ntau - 1):

        current_Bstring = current_Bstring @ Bs[:,:,i]

        temp[:2*N,:2*N] = (np.identity(2*N) - current_Bstring) @ scipy.linalg.inv(np.identity(2*N) + current_Bstring)
        temp[:2*N,2*N:] = -np.identity(2*N)
        temp[2*N:,:2*N] = np.identity(2*N)
        temp[2*N:,2*N:] = scipy.linalg.tanhm(0.5*Alist[i+1])

        pf_prod *= pf.pfaffian(temp)

    W = np.power(2, N) * np.power(-1, (Ntau + 1)*N) *eta_prod * pf_prod

    return W

def generate_random_Alist(N, Ntau, scaling = 0.1):

    Alist = []
    rng = np.random.default_rng(seed = 412425)

    for i in range(Ntau):

        A = rng.normal(loc = 0, scale = scaling, size = (2*N, 2*N)) + 1j * rng.normal(loc = 0, scale = scaling, size = (2*N, 2*N))
        A = 0.5*(A - A.T)
        Alist.append(A)

    return Alist

def get_J_cross(matrix):

    dim = matrix.shape[0]

    Jcross = np.zeros((2*dim, 2*dim), dtype = matrix.dtype)

    Jcross[:dim, dim:2*dim] = - matrix
    Jcross[dim:2*dim, :dim] = + matrix

    return Jcross

def cos_string(phi,dtau):

    prod = 1

    for i in range(phi.size):

        prod = prod * np.cos(phi[i]*dtau) * (-1)

    return prod


class HMC():

    def __init__(self, t_MD, N_MD, model, saver, seed = 1337):
        self.N_MD = N_MD
        self.t_MD  = t_MD
        self.dt, self.dt_half = t_MD/self.N_MD, 0.5*t_MD/self.N_MD
        self.model = model
        self.cfg_shape = model.shape
        self.saver = saver
        self.steps = self.acc = 0
        self.rng   = np.random.default_rng(seed)


    def step(self):

        self.mom = self.rng.normal(loc = 0.0, scale = 1.0, size = self.cfg_shape).astype(np.complex128)

        new_cfg  = np.copy(self.model.cfg)

        old_H = self.model.action + 0.5 * np.sum(np.power(self.mom, 2))

        self.mom -= self.dt_half*self.model.get_action_gradient(new_cfg)

        for _ in range(self.N_MD-1):

            new_cfg +=self.dt*self.mom
            self.mom -= self.dt*self.model.get_action_gradient(new_cfg)
        
        new_cfg += self.dt*self.mom
        self.mom -= self.dt_half*self.model.get_action_gradient(new_cfg)

        new_action = self.model.get_action_catterpiller(new_cfg)

        new_H = new_action + 0.5* np.sum(np.power(self.mom, 2))


        dH    = new_H - old_H

        p = np.exp(-dH)
        r = self.rng.random()

        if r < p :

            self.acc += 1

            self.model.cfg = new_cfg
            self.model.action = new_action

        self.steps += 1


    def save(self):
        self.model.update_obs()
        self.saver.save_cfg_and_obs(self.model.cfg, self.model.obs_dict)

    def output_acc(self):
        return self.acc/self.steps
        
    def reset_acc(self):
        self.acc = self.steps = 0

    def output_model(self):
        return self.model

    def close_file(self):
        self.saver.close()



class Kitaev_action():
    def __init__(self, L, U, Delta, mu, dtau, Ntau, seed = 153525):
        #set paramters and initialize configuration     
        self.U = U

        self.mu = mu
        self.dtau = dtau
        self.Ntau = Ntau
        self.Delta = Delta
        self.L = L
        self.Dplus = 1 + Delta
        self.Dminus = 1 - Delta

        self.rng = np.random.default_rng(seed)
        self.shape = (L, Ntau)
        self.cfg = self.rng.normal(loc=0.0, scale = 1/np.sqrt(np.prod(self.shape)), size = self.shape).astype(np.complex128)
        self.gradient = np.zeros((self.cfg.shape), dtype=self.cfg.dtype)
        self.J = np.array([[0,-1],[1,0]])
        self.I = np.eye(2)
        self.eye = np.eye(2*self.L, dtype = np.complex128)

        self.A_odd = np.zeros((2*L, 2*L), dtype = np.complex128)
        self.A_odd[:L,:L] = np.roll(np.eye(L)*self.Dminus, 1, axis = 1) - np.roll(np.eye(L)*self.Dminus, -1, axis = 1)
        self.A_odd[L:,L:] = np.roll(np.eye(L)*self.Dplus, 1, axis = 1) - np.roll(np.eye(L)*self.Dplus, -1, axis = 1)

        self.A_odd[:L,L:] = np.eye(L)*mu
        self.A_odd[L:,:L] = -np.eye(L)*mu

        self.A_odd[L-1, 0] = 0
        self.A_odd[0, L-1] = 0

        self.A_odd[2*L - 1, L] = 0
        self.A_odd[L, 2*L-1]   = 0

        self.A_odd = self.A_odd * self.dtau * 1j
        self.eta_odd, self.B_odd = get_eta_B(self.A_odd)


        self.G_odd = scipy.linalg.tanhm(self.A_odd/2)
        self.expminusA = scipy.linalg.expm(-self.A_odd)
        self.expplusA  = scipy.linalg.expm(self.A_odd)
        self.eta_odd_prod = np.power(self.eta_odd, self.Ntau)

        self.temp = np.zeros((4*self.L, 4*self.L), dtype = np.complex128)
        self.temp[2*self.L:, :2*self.L] = np.eye(2*self.L)
        self.temp[:2*self.L, 2*self.L:] = -np.eye(2*self.L)

        self.g = np.zeros((self.Ntau*self.L*4, self.Ntau*self.L*4))

        block_size = 2 * L
        total_size = block_size * Ntau *2

        self.g = np.zeros((total_size, total_size), dtype=np.complex128)
        I = np.eye(block_size, dtype=np.complex128)

        for i in range(2*Ntau):
            for j in range(i + 1, 2*Ntau):
                # Checkerboard sign on upper triangle:
                # starts with minus for (i,j) = (0,1)
                sign = -1 if (j - i) % 2 == 1 else 1

                row_i = slice(i * block_size, (i + 1) * block_size)
                row_j = slice(j * block_size, (j + 1) * block_size)

                self.g[row_i, row_j] = sign * I
                self.g[row_j, row_i] = -sign * I  # antisymmetry

            

        self.action = self.get_action_catterpiller(self.cfg)

    def get_expminusAeven(self, cfg_t):

        return (np.kron(self.I, np.diag(np.cos(2*self.dtau*cfg_t[:]))) - np.kron(self.J, np.diag(np.sin(2*self.dtau*cfg_t[:]))))
   
   
    def get_expplusAeven(self, cfg_t):

            return (np.kron(self.I, np.diag(np.cos(2*self.dtau*cfg_t[:]))) + np.kron(self.J, np.diag(np.sin(2*self.dtau*cfg_t[:]))))

    def get_all_G_even(self, cfg):

        all_G_even = np.zeros((2*self.L, 2*self.L, self.Ntau), dtype =  np.complex128)

        temp = (np.eye(self.cfg.shape[0], dtype=np.complex128) * np.tan(self.dtau*cfg.T[:, None, :])).transpose(1, 2, 0)

        all_G_even[self.L:, :self.L,:] = temp
        all_G_even[:self.L, self.L:,:] = -temp
        
        return all_G_even
        

    def get_all_As(self, cfg):

        allAs = np.zeros((2*self.Ntau,self.L*2, self.L*2), dtype = np.complex128)

        for i in range(self.Ntau):

            allAs[2*i,:,:] = self.A_odd.copy()
            allAs[2*i+1,:,:] = np.kron(self.J, np.diag(2*self.dtau*cfg[:,i]))


        return allAs
    
    def get_g(self, cfg):

        block_size = 2*self.L
        I = np.eye(block_size, dtype=np.complex128)

        for j in range(self.Ntau):

                row_j = slice(2*j * block_size, (2*j + 1) * block_size)

                self.g[row_j, row_j] = self.G_odd
                
                #even case
                row_j = slice((2*j+1) * block_size, ((2*j+1) + 1) * block_size)
                self.g[row_j, row_j] = np.kron(self.J, np.diag(np.tan(self.dtau * cfg[:,j])))

                # plt.imshow(np.abs(self.g[row_j, row_j]))
                # plt.show()

    def get_action_catterpiller(self, cfg):

        action_B  = np.sum(2*cfg*cfg)/(self.dtau*self.U) # auxiliarry action


        allBs   = np.zeros((2*self.L, 2*self.L, self.Ntau), dtype = np.complex128)
        alletas = np.zeros(self.Ntau, dtype = np.complex128)
        allGs   = np.zeros((2*self.L, 2*self.L, self.Ntau), dtype = np.complex128)

        current_G = self.G_odd.copy()
        current_B = self.expminusA.copy()
        current_eta = self.eta_odd

        I2 = np.eye(2*self.L, dtype=np.complex128)

        for i in range(self.Ntau):
            
            allBs[:,:,i]   = np.kron(self.I, np.diag(np.cos(2 * self.dtau * cfg[:,i]))) - np.kron(self.J, np.diag(np.sin(2*self.dtau*cfg[:,i])))
            alletas[i] = cos_string(cfg[:,i], self.dtau)
            allGs[:,:,i]   = np.kron(self.J, np.diag(np.tan(self.dtau * cfg[:,i])))

            self.temp[:2*self.L, :2*self.L] = current_G
            self.temp[2*self.L:, 2*self.L:] = allGs[:,:,i]
            current_eta                     = np.power(-1, self.L) * current_eta * alletas[i] * pf.pfaffian(self.temp)
            current_B                       = current_B @ allBs[:,:,i]
            current_G                       = (I2 - current_B)@scipy.linalg.inv(I2 + current_B)

            # print("orth err =", np.linalg.norm(current_B.T @ current_B - I2))
            # print("skew err =", np.linalg.norm(current_G + current_G.T))
            # print("cond(I+B) =", np.linalg.cond(I2 + current_B))
            # print("min |eig(B)+1| =", np.min(np.abs(np.linalg.eigvals(current_B) + 1)))

            if i != (self.Ntau - 1):
                self.temp[:2*self.L, :2*self.L] = current_G
                self.temp[2*self.L:, 2*self.L:] = self.G_odd
                current_eta                     = np.power(-1, self.L) * current_eta * self.eta_odd * pf.pfaffian(self.temp)
                current_B                       = current_B @ self.expminusA
                current_G                       = (I2 - current_B)@scipy.linalg.inv(I2 + current_B)

            # print("orth err =", np.linalg.norm(current_B.T @ current_B - I2))
            # print("skew err =", np.linalg.norm(current_G + current_G.T))
            # print("cond(I+B) =", np.linalg.cond(I2 + current_B))
            # print("min |eig(B)+1| =", np.min(np.abs(np.linalg.eigvals(current_B) + 1)))

        weight = 2**self.L * current_eta

        #This looks odd because in the case of the Kitaev chain and the way I HS-transformed A^up and A^down are equal, therefore the log must appear twice.
        # Perhaps a different HS-Trafo would be better
        self.action = action_B - np.log(weight) - np.log(weight)

        return self.action
    
    def get_action_gradient(self, cfg):

        # W = self.get_action_catterpiller(cfg)
        gradient_p1 = -self.dtau * np.tan(self.dtau * cfg)

        self.get_g(cfg)

        g_inv = scipy.linalg.inv(self.g)

        t = np.arange(self.Ntau)[:, None]
        x = np.arange(self.L)[None, :]

        rows = (2*t + 1) * (2*self.L) + x
        cols = rows + self.L

        g_inv_diag = g_inv[rows, cols].reshape(self.Ntau, self.L)

        gradient_p2 = self.dtau * g_inv_diag.T*(1 + np.power(np.tan(self.dtau*cfg), 2))

        gradient_S_B = 4*cfg/(self.dtau * self.U)

        gradient = -(gradient_p1 + gradient_p2) - (gradient_p1 + gradient_p2) + gradient_S_B

        return gradient
    
    def get_action_gradient_catterpiller(self, cfg):

        # W = self.get_action_catterpiller(cfg)
        gradient = +2*self.dtau * np.tan(self.dtau * cfg)

        gradient_S_B = 4*cfg/(self.dtau * self.U)

        S = np.eye(2*self.L)

        # print(S.shape)
        # print(self.expminusA.shape)
        # print(self.I)
        # print(np.kron(self.I, np.diag(np.cos(2*self.dtau*cfg[:,0]))).shape)

        Gks = self.get_all_G_even(cfg)

        for i in range(self.Ntau):
            print(i)
            S = S @ self.expminusA
            S = S @ (np.kron(self.I, np.diag(np.cos(2*self.dtau*cfg[:,i]))) - np.kron(self.J, np.diag(np.sin(2*self.dtau*cfg[:,i]))))

        currentB = S.copy()
        B_i = np.eye(self.L*2)

        for i in range(self.Ntau):

            currentB = self.expplusA @ currentB @ B_i

            B_i      = self.expminusA

            currentB = self.get_expplusAeven(cfg[:, i]) @ currentB @ B_i

            B_i      = self.get_expminusAeven(cfg[:,i])

            Denominator = self.eye + currentB
            Enumerator  = self.eye - currentB

            InverseFactor = scipy.linalg.inv(Gks[:,:,i]@Enumerator + Denominator)

            temp_grad = Enumerator @ InverseFactor
            temp_grad = np.diagonal(temp_grad[:self.L,self.L:]) * self.dtau * (1 + np.power(np.tan(self.dtau*cfg[:,i]),2))
        
            gradient[:,i] = gradient[:,i] - 2*temp_grad.copy()


        gradient += gradient_S_B
            
        return gradient



In [197]:
L = 6
U = 1
Delta = 0.2
mu = 2
dtau = 0.1
Ntau = 2

model = Kitaev_action(L, U, Delta, mu, dtau, Ntau)
maction = model.get_action_catterpiller(model.cfg)

allAs = model.get_all_As(model.cfg)

grad = model.get_action_gradient(model.cfg)

grad_cat = model.get_action_gradient_catterpiller(model.cfg)

print(grad)
print(grad_cat)

print(grad - grad_cat)
print(np.allclose(grad, grad_cat))

0
1
[[ 1.13524971e+01-0.03910596j -4.84187309e+00-0.03910532j]
 [-8.73111108e-01-0.03872505j  6.00064699e+00-0.03872484j]
 [ 8.38627129e-04-0.03880535j -1.93445090e+01-0.0388086j ]
 [-1.28969362e+01-0.03876838j -1.51976027e+00-0.03876632j]
 [-1.34945291e+01-0.03873801j  4.23563913e+00-0.03873648j]
 [-4.38455946e+00-0.03913284j -8.38939561e+00-0.03913392j]]
[[ 1.13524971e+01-0.03910596j -4.84187309e+00-0.03910532j]
 [-8.73111108e-01-0.03872505j  6.00064699e+00-0.03872484j]
 [ 8.38627129e-04-0.03880535j -1.93445090e+01-0.0388086j ]
 [-1.28969362e+01-0.03876838j -1.51976027e+00-0.03876632j]
 [-1.34945291e+01-0.03873801j  4.23563913e+00-0.03873648j]
 [-4.38455946e+00-0.03913284j -8.38939561e+00-0.03913392j]]
[[0.00000000e+00-2.08166817e-17j 0.00000000e+00-6.24500451e-17j]
 [0.00000000e+00-6.93889390e-18j 0.00000000e+00-1.38777878e-17j]
 [1.73472348e-18+3.46944695e-17j 0.00000000e+00+0.00000000e+00j]
 [0.00000000e+00-6.93889390e-18j 0.00000000e+00-1.38777878e-17j]
 [0.00000000e+00+2.0816681

In [23]:
L = 6
U = 1
Delta = 0.2
mu = 0
dtau = 0.1
Ntau = 2
t_MD = 0.8
N_MD = 5


model = Kitaev_action(L, U, Delta, mu, dtau, Ntau)
algorithm = HMC(t_MD, N_MD, model, saver = 0, seed = 3432)

for i in range(1000):
    algorithm.step()

algorithm.reset_acc()

for i in range(1000):
    algorithm.step()
algorithm.output_acc()

(0.012991355684371634+0j)
(18.975119456249157+0j)
(50.49082811842678+0j)
(22.394501567020654+0j)
(146.34088268946695+0j)
(11.38725965796313+0j)
(0.35139332500731557+0j)
(1.2490171718147596+0j)
(2.3225998820669074+0j)
(4.646793991086883+0j)
(0.1861728925158645+0j)
(0.09496711294216137+0j)
(2.861911044063794+0j)
(0.06816926977906503+0j)
(52.10536286579895+0j)
(4.548868789648664+0j)
(9.573240286231352+0j)
(14.319047351845157+0j)
(273.0214898024149+0j)
(0.2455159247758591+0j)
(37.01977259947918+0j)
(2.077753230862196+0j)
(0.5089073591068759+0j)
(109.82587732352182+0j)
(14.432907739314293+0j)
(152.5824306658206+0j)
(77.07612195928208+0j)
(235.7335125598027+0j)
(7.205299828720256e-05+0j)
(0.7027758798847016+0j)
(0.2981604083838533+0j)
(2.2544759744989724+0j)
(234.55139724434196+0j)
(3.3597359509332674+0j)
(150.062612236435+0j)
(10.354939051019826+0j)
(0.5417934280325853+0j)
(0.09070806767361272+0j)
(0.2943758977306183+0j)
(0.08475393801359869+0j)
(1.0939482327996597+0j)
(6.505480226617017+0j

0.789

In [ ]:
N = 4
dtau = 0.2
rng = np.random.default_rng(4135)
phi = rng.normal(loc = 0.0, scale = 1, size = (N))
temp = np.diag(2*dtau*phi)

A = get_J_cross(temp)

eta, B = get_eta_B(A)
eta_fast = cos_string(phi, dtau)

print(eta)
print(eta_fast)


In [ ]:
catterpillar(Alist)

In [ ]:
brute_force_trace(Alist)

In [ ]:
recursive_eta(Alist)

In [58]:
def get_G(A):

    D = scipy.linalg.inv(np.eye(A.shape[0]) + A)
    Nu= np.eye(A.shape[0]) - A

    return D @ Nu

In [67]:
Alist = generate_random_Alist(2, 3)

eta_1, B1 = get_eta_B(Alist[0])
eta_2, B2 = get_eta_B(Alist[1])
eta_3, B3 = get_eta_B(Alist[2])

eta_12, B12 = multiplication(eta_1, B1, eta_2, B2)
eta_21, B21 = multiplication(eta_2, B2, eta_1, B1)


G = get_G(B12)
Ginv = scipy.linalg.inv(G)
Gmb = get_G(-B12)


print(Ginv-Gmb)
print(Gmb)


[[-9.76996262e-15+8.93729535e-15j -8.43769499e-15+0.00000000e+00j
   6.21724894e-15+5.32907052e-15j -8.88178420e-15-5.32907052e-15j]
 [-1.77635684e-15-1.06581410e-14j  5.32907052e-15+2.10942375e-15j
   8.88178420e-16+3.55271368e-15j -1.77635684e-15+5.32907052e-15j]
 [ 3.55271368e-15-3.55271368e-15j  2.66453526e-15-3.55271368e-15j
  -2.73298669e-15-2.61028144e-15j  3.55271368e-15+3.21964677e-15j]
 [-3.55271368e-15+0.00000000e+00j  0.00000000e+00+1.77635684e-15j
   0.00000000e+00+2.77555756e-15j -3.66373598e-15-6.66133815e-16j]]
[[ 5.32907052e-15-8.93729535e-15j  2.89730090e+00-2.80885611e+01j
  -6.86373882e+00-1.06029205e+01j  9.15212804e+00-8.52026845e+00j]
 [-2.89730090e+00+2.80885611e+01j  4.08562073e-14+1.25455202e-14j
   4.55891552e+00-1.64369006e+01j -4.17989506e+00+9.24370871e+00j]
 [ 6.86373882e+00+1.06029205e+01j -4.55891552e+00+1.64369006e+01j
   2.55524768e-15-6.99440506e-15j -1.58647439e+01+7.62096081e-01j]
 [-9.15212804e+00+8.52026845e+00j  4.17989506e+00-9.24370871e+00j
  

In [70]:
Alist = generate_random_Alist(2, 4)

eta_1, B1 = get_eta_B(Alist[0])
eta_2, B2 = get_eta_B(Alist[1])
eta_3, B3 = get_eta_B(Alist[2])
eta_4, B4 = get_eta_B(Alist[3])

eta_12, B12 = multiplication(eta_1, B1, eta_2, B2)
eta_123, B123 = multiplication(eta_12, B12, eta_3,B3)
eta_1234, B1234 = multiplication(eta_123, B123, eta_4, B4)

eta_34, B34 = multiplication(eta_3, B3, eta_4, B4)
eta_341, B341 = multiplication(eta_34, B34, eta_1, B1)
eta_3412, B3412 = multiplication(eta_341, B341, eta_2, B2)

print(eta_3412)
print(eta_1234)

(0.9908392818070638-0.008791531664595042j)
(0.9908392818070639-0.008791531664595037j)


In [188]:

L = 6
U = 1
Delta = 0.2
mu = 0
dtau = 0.1
Ntau = 2
t_MD = 0.8
N_MD = 5


model = Kitaev_action(L, U, Delta, mu, dtau, Ntau)

grad_1 = model.get_action_gradient_catterpiller(model.cfg)
grad_2 = model.get_action_gradient(model.cfg)

print(grad_1)
print(grad_2)

0
1
[[ 1.13441073e+01+0.j -4.83811917e+00+0.j]
 [-8.72360641e-01+0.j  5.99625108e+00+0.j]
 [ 4.79467719e-04+0.j -1.93303588e+01+0.j]
 [-1.28875276e+01+0.j -1.51888313e+00+0.j]
 [-1.34845748e+01+0.j  4.23229747e+00+0.j]
 [-4.38151383e+00+0.j -8.38334968e+00+0.j]]
[[ 1.13526216e+01+0.j -4.84175304e+00+0.j]
 [-8.73017239e-01+0.j  6.00074997e+00+0.j]
 [ 4.86888909e-04+0.j -1.93448715e+01+0.j]
 [-1.28971985e+01+0.j -1.52001789e+00+0.j]
 [-1.34946962e+01+0.j  4.23547753e+00+0.j]
 [-4.38479785e+00+0.j -8.38963804e+00+0.j]]


In [86]:
model.cfg[:,:] = np.arange(0, model.L*model.Ntau).reshape(model.L,model.Ntau)
out = (np.eye(model.cfg.shape[0], dtype=model.cfg.dtype) * model.cfg.T[:, None, :]).transpose(1, 2, 0)
print(model.cfg)
print(out[:,:,1])

[[ 0.+0.j  1.+0.j]
 [ 2.+0.j  3.+0.j]
 [ 4.+0.j  5.+0.j]
 [ 6.+0.j  7.+0.j]
 [ 8.+0.j  9.+0.j]
 [10.+0.j 11.+0.j]]
[[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  3.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  5.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  7.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  0.+0.j  9.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j 11.+0.j]]


In [143]:
Alist = model.get_all_As(model.cfg)

eta_list = np.zeros(2*model.Ntau, dtype = np.complex128)
Blist    = np.zeros((2*model.L, 2*model.L, model.Ntau*2), dtype = np.complex128)

for i in range(2*Ntau):
    eta_list[i], Blist[:,:,i] = get_eta_B(Alist[i])


eta_c, Bc = eta_list[0], Blist[:,:,0]

for i in range(2*model.Ntau-1):

    eta_c, Bc = multiplication(eta_c, Bc, eta_list[i+1], Blist[:,:,i+1])


k = 2 #0 in code is 1 on paper


eta_k, Bk = eta_list[(k+1)%(2*model.Ntau)], Blist[:,:,(k+1) % (2*model.Ntau)]

for i in range(2*model.Ntau-1):
    print((k+i+2)%(2*model.Ntau))
    eta_k, Bk = multiplication(eta_k, Bk, eta_list[(k+i+2)%(2*model.Ntau)], Blist[:,:,(k+i+2)%(2*model.Ntau)])


print(eta_c*np.power(2, model.L))
print(eta_k*np.power(2, model.L))
print(recursive_eta(Alist))


4
5
6
7
8
9
10
11
12
13
14
15
0
1
2
(137.61697535069834+0j)
(137.61697535069862+0j)
(137.61697535069834+0j)


In [193]:
def grad(sself, cfg):
        
        sself.get_g(cfg)

        g_inv = scipy.linalg.inv(sself.g)

        t = np.arange(sself.Ntau)[:, None]
        x = np.arange(sself.L)[None, :]

        rows = (2*t + 1) * (2*sself.L) + x
        cols = rows + sself.L

        g_inv_diag = g_inv[rows, cols].reshape(sself.Ntau, sself.L)

        gradient_p2 = sself.dtau * g_inv_diag.T*(1 + np.power(np.tan(sself.dtau*cfg), 2))

        return gradient_p2

def grad_c(self, cfg):

        S = np.eye(2*self.L)

        Gks = self.get_all_G_even(cfg)

        gradient = cfg.copy()*0

        for i in range(self.Ntau):

            S = S @ self.expminusA
            S = S @ (np.kron(self.I, np.diag(np.cos(2*self.dtau*cfg[:,i]))) - np.kron(self.J, np.diag(np.sin(2*self.dtau*cfg[:,i]))))

        currentB = S.copy()
        B_i = np.eye(self.L*2)

        for i in range(self.Ntau):

            currentB = self.expplusA @ currentB @ B_i

            B_i      = self.expminusA

            currentB = self.get_expplusAeven(cfg[:, i]) @ currentB @ B_i

            B_i      = self.get_expminusAeven(cfg[:,i])

            Denominator = self.eye + currentB
            Enumerator  = self.eye - currentB

            InverseFactor = scipy.linalg.inv(Gks[:,:,i]@Enumerator + Denominator)

            temp_grad = Enumerator @ InverseFactor
            temp_grad = np.diagonal(temp_grad[:self.L,self.L:]) * self.dtau * (1 + np.power(np.tan(self.dtau*cfg[:,i]),2))

            gradient[:,i] = temp_grad.copy()            


        return gradient

In [194]:
L = 6
U = 1
Delta = 0.2
mu = 0
dtau = 0.1
Ntau = 2
t_MD = 0.8
N_MD = 5


model = Kitaev_action(L, U, Delta, mu, dtau, Ntau)

g1 = grad(model, model.cfg)
g2 = grad_c(model, model.cfg)

print(g1)
print(g2)

print(np.allclose(g1, g2))

[[ 1.19743871e-03+0.j -2.79420269e-03+0.j]
 [-1.44204260e-03+0.j  1.57536555e-04+0.j]
 [ 4.70648917e-03+0.j  8.15146901e-06+0.j]
 [ 3.39250125e-04+0.j  3.12276064e-03+0.j]
 [-1.07974545e-03+0.j  3.28054436e-03+0.j]
 [ 2.04337856e-03+0.j  1.09426710e-03+0.j]]
[[ 1.19743871e-03+0.j -2.79420269e-03+0.j]
 [-1.44204260e-03+0.j  1.57536555e-04+0.j]
 [ 4.70648917e-03+0.j  8.15146901e-06+0.j]
 [ 3.39250125e-04+0.j  3.12276064e-03+0.j]
 [-1.07974545e-03+0.j  3.28054436e-03+0.j]
 [ 2.04337856e-03+0.j  1.09426710e-03+0.j]]
True
